In [ ]:
"""
Q-Learning LQR — LS + GD (Algoritmo B.1)
Transcripción fiel del código Octave al Apéndice B.
"""
import sys
import numpy as np
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget,
    QVBoxLayout, QHBoxLayout, QLabel, QPushButton, QSizePolicy
)
from PyQt6.QtCore import QTimer, Qt
from PyQt6.QtGui import QFont
import pyqtgraph as pg


# ─────────────────────────────────────────────────────────────
# Función michirp  — igual que el .m de referencia
# ─────────────────────────────────────────────────────────────
def sat(x):
    return np.clip(x, -1.0, 1.0)

def michirp(wini, wfin, N, T):
    """Genera la señal chirp de excitación persistente (N filas, 2 cols)."""
    u = np.zeros((N, 2))
    for i in range(1, N + 1):
        wk    = wini + (i - 1) / N * (wfin - wini)
        w_env = sat(10.0 * i / N) * sat((N - i) / (0.1 * N))
        u[i - 1, 0] = i * T
        u[i - 1, 1] = w_env * np.sin(wk * i * T)
    return u


# ─────────────────────────────────────────────────────────────
# Ventana principal
# ─────────────────────────────────────────────────────────────
class LQR_RL_App(QMainWindow):

    # ── constantes del algoritmo (Octave líneas 8-60) ─────────────────────
    A   = np.array([[1.8980, -0.9048], [1.0, 0.0]])
    B   = np.array([[1.0], [0.0]])
    Q_m = np.diag([1.0, 1.0])
    R_v = 2.0
    N_MAX      = 4000
    GAMMA      = 0.999
    N_LS       = 7          # tamaño de lote LS
    ALPHA      = 120.02     # tasa aprendizaje GD
    B_GAIN     = 1.0        # ganancia de excitación
    CR         = 0.202      # escala chirp
    CON_ERR_K  = 10e-2      # = 0.1  (Octave línea 54)

    # H inicial (Octave línea 24)
    H0 = np.array([[14., -2.,  2.],
                   [-8.,  3., -1.],
                   [ 8., -5.,  4.]], dtype=float)

    # Valores objetivo de W para líneas de referencia
    W_TARGET = [17.2962, -8.6202, 9.5272, 6.0227, -5.5512, 8.1353]

    def __init__(self):
        super().__init__()
        self.setWindowTitle("Q-Learning LQR — LS + GD  (Algoritmo B.1)")
        self.setStyleSheet("background:#0d0f14; color:#e0e0e0;")

        # Señal de excitación (N+1 muestras, usa índices 0..N-1)
        self.inp = michirp(0.005, 3_450_000, self.N_MAX + 1, 0.1)

        self._reset_state()
        self._build_ui()

        self.timer = QTimer()
        self.timer.timeout.connect(self._step)
        self.timer.start(0)     # 0 ms → corre lo más rápido posible

    # ── Estado interno ────────────────────────────────────────────────────
    def _reset_state(self):
        self.H       = self.H0.copy()
        self.K       = self._K_from_H(self.H)
        self.x       = np.array([[5.0], [-4.0]])
        self.W_H_ant = np.zeros((6, 1))

        self.phi_ls  = np.zeros((6, self.N_LS))
        self.phi1_ls = np.zeros((6, self.N_LS))
        self.r_ls    = np.zeros((self.N_LS, 1))

        self.i           = 1    # iteración 1-based (igual que Octave)
        self.iteracion   = 0    # conteo de actualizaciones de H y K

        # historiales para gráficas
        self.w_hist  = [[] for _ in range(6)]
        self.x1_hist = []

    # ── K desde H (Octave líneas 27-29) ──────────────────────────────────
    @staticmethod
    def _K_from_H(H):
        Huu = H[2, 2]
        Hux = H[2, 0:2]
        return -(1.0 / Huu) * Hux.reshape(1, 2)

    # ── Construcción de la GUI ────────────────────────────────────────────
    def _build_ui(self):
        pg.setConfigOptions(antialias=True,
                            background='#0d0f14',
                            foreground='#e0e0e0')

        cw = QWidget()
        self.setCentralWidget(cw)
        root = QHBoxLayout(cw)
        root.setSpacing(8)
        root.setContentsMargins(8, 8, 8, 8)

        # ── Panel izquierdo: convergencia de W ────────────────────────────
        left = QVBoxLayout()
        left.setSpacing(4)

        lbl_w = QLabel("Convergencia de W (parámetros de H)")
        lbl_w.setFont(QFont("Courier New", 9))
        lbl_w.setStyleSheet("color:#88ccff;")
        lbl_w.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)
        left.addWidget(lbl_w)

        self.plot_w = pg.PlotWidget()
        self.plot_w.setLabel('left',   'W')
        self.plot_w.setLabel('bottom', 'iteración')
        self.plot_w.addLegend(offset=(5, 5))
        self.plot_w.setSizePolicy(QSizePolicy.Policy.Expanding,
                                  QSizePolicy.Policy.Expanding)

        colores = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff', '#c77dff', '#ff9a3c']
        nombres = ['W1', 'W2', 'W3', 'W4', 'W5', 'W6']
        self.w_curves = []
        for j in range(6):
            c = self.plot_w.plot([], pen=pg.mkPen(colores[j], width=1.6),
                                 name=nombres[j])
            self.w_curves.append(c)
            ref = pg.InfiniteLine(
                pos=self.W_TARGET[j], angle=0,
                pen=pg.mkPen(colores[j], style=Qt.PenStyle.DashLine, width=1)
            )
            self.plot_w.addItem(ref)
        left.addWidget(self.plot_w, 3)

        root.addLayout(left, 2)

        # ── Panel derecho ─────────────────────────────────────────────────
        right = QVBoxLayout()
        right.setSpacing(4)

        lbl_x = QLabel("Estado  x₁  (posición de la masa)")
        lbl_x.setFont(QFont("Courier New", 9))
        lbl_x.setStyleSheet("color:#88ccff;")
        lbl_x.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)
        right.addWidget(lbl_x)

        self.plot_x = pg.PlotWidget()
        self.plot_x.setLabel('left',   'x₁')
        self.plot_x.setLabel('bottom', 'muestra')
        self.plot_x.setSizePolicy(QSizePolicy.Policy.Expanding,
                                  QSizePolicy.Policy.Expanding)
        self.curve_x = self.plot_x.plot([], pen=pg.mkPen('#ffd93d', width=1.6))
        right.addWidget(self.plot_x, 2)

        # info numérica
        self.lbl_info = QLabel("Iniciando…")
        self.lbl_info.setFont(QFont("Courier New", 9))
        self.lbl_info.setStyleSheet(
            "background:#0a0d18; color:#00ff88; padding:8px; border:1px solid #223;"
        )
        self.lbl_info.setAlignment(
            Qt.AlignmentFlag.AlignLeft | Qt.AlignmentFlag.AlignTop
        )
        self.lbl_info.setSizePolicy(QSizePolicy.Policy.Preferred,
                                    QSizePolicy.Policy.Fixed)
        right.addWidget(self.lbl_info)

        btn = QPushButton("⟳  REINICIAR")
        btn.setStyleSheet(
            "background:#1a2a4a; color:#88ccff; font-size:12px; "
            "padding:6px; border:1px solid #335; border-radius:4px;"
        )
        btn.setSizePolicy(QSizePolicy.Policy.Preferred, QSizePolicy.Policy.Fixed)
        btn.clicked.connect(self._on_reset)
        right.addWidget(btn)

        root.addLayout(right, 1)

    def _on_reset(self):
        self._reset_state()
        for c in self.w_curves:
            c.setData([], [])
        self.curve_x.setData([], [])
        self.lbl_info.setText("Reiniciado.")
        if not self.timer.isActive():
            self.timer.start(0)

    # ── Paso: corre N_BATCH iteraciones por tick para no bloquear la GUI ──
    N_BATCH = 50

    def _step(self):
        for _ in range(self.N_BATCH):
            if self.i > self.N_MAX:
                self.timer.stop()
                break
            self._iteration()
        self._update_plots()

    # ── Una iteración del bucle Octave `for i = 1:N` ──────────────────────
    def _iteration(self):
        i  = self.i
        xi = self.x.copy()       # x(:,i)

        # control (Octave línea 73)
        u_sc = (float((self.K @ xi).item())
                + self.B_GAIN * float(self.inp[i - 1, 1]) * self.CR)

        # dinámica (Octave línea 74)
        xn = self.A @ xi + self.B * u_sc    # x(:,i+1)

        # u sin excitación para el siguiente estado (Octave línea 76)
        u2 = float((self.K @ xn).item())

        # buffer circular LS (Octave líneas 76-82)
        self.r_ls    = np.roll(self.r_ls,    -1, axis=0)
        self.phi_ls  = np.roll(self.phi_ls,  -1, axis=1)
        self.phi1_ls = np.roll(self.phi1_ls, -1, axis=1)

        self.r_ls[-1, 0] = (float((xi.T @ self.Q_m @ xi).item())
                            + u_sc ** 2 * self.R_v)

        self.phi_ls[:, -1] = [
            xi[0,0]**2, xi[0,0]*xi[1,0], xi[0,0]*u_sc,
            xi[1,0]**2, xi[1,0]*u_sc,    u_sc**2
        ]
        self.phi1_ls[:, -1] = [
            xn[0,0]**2, xn[0,0]*xn[1,0], xn[0,0]*u2,
            xn[1,0]**2, xn[1,0]*u2,       u2**2
        ]

        # inicialización LS (Octave líneas 84-89)
        if i == self.N_LS:
            V = self.phi_ls - self.phi1_ls          # (6 x n_ls)
            # lstsq equivale numéricamente a inv(V*V')*V*r de Octave
            # (Octave usa LU implícito; inv() sobre C_ singular daría overflow)
            self.W_H_ant, _, _, _ = np.linalg.lstsq(V.T, self.r_ls, rcond=None)

        # GD (Octave líneas 91-131)
        if i > self.N_LS:
            # phi y phi1 frescos del instante i (Octave líneas 93-96)
            rk = (float((xi.T @ self.Q_m @ xi).item()) + u_sc ** 2 * self.R_v)

            phi_k = np.array([
                xi[0,0]**2, xi[0,0]*xi[1,0], xi[0,0]*u_sc,
                xi[1,0]**2, xi[1,0]*u_sc,    u_sc**2
            ]).reshape(6, 1)

            phi1_k = np.array([
                xn[0,0]**2, xn[0,0]*xn[1,0], xn[0,0]*u2,
                xn[1,0]**2, xn[1,0]*u2,       u2**2
            ]).reshape(6, 1)

            vec_act = phi_k - self.GAMMA * phi1_k

            # paso GD (Octave línea 97)
            W_H = self.W_H_ant - self.ALPHA * vec_act * (
                float((self.W_H_ant.T @ vec_act).item()) - rk
            )

            # error TD (Octave línea 98)
            e_d_t = -(float((W_H.T @ vec_act).item())) + rk

            # condición de actualización (Octave línea 99)
            if abs(e_d_t) < self.CON_ERR_K:
                self.iteracion += 1
                W = W_H.flatten()
                self.H = np.array([
                    [W[0],    W[1]/2,  W[2]/2],
                    [W[1]/2,  W[3],    W[4]/2],
                    [W[2]/2,  W[4]/2,  W[5]  ]
                ])
                self.K       = self._K_from_H(self.H)
                self.W_H_ant = W_H.copy()

        # guardar historiales
        for j in range(6):
            self.w_hist[j].append(float(self.W_H_ant[j, 0]))
        self.x1_hist.append(float(xn[0, 0]))

        self.x = xn
        self.i += 1

    # ── Actualizar gráficas ────────────────────────────────────────────────
    def _update_plots(self):
        xs = list(range(len(self.w_hist[0])))
        for j in range(6):
            self.w_curves[j].setData(xs, self.w_hist[j])

        self.curve_x.setData(self.x1_hist[-800:])

        W  = self.W_H_ant.flatten()
        Ke = -self.K        # K_e = -K  (Octave línea 106)
        done = "✓ LISTO" if self.i > self.N_MAX else ""
        self.lbl_info.setText(
            f"  iter: {self.i - 1} / {self.N_MAX}   actualizaciones: {self.iteracion}  {done}\n"
            f"  K  = [{self.K[0,0]:+.4f}  {self.K[0,1]:+.4f}]\n"
            f"  K_e= [{Ke[0,0]:+.4f}  {Ke[0,1]:+.4f}]\n"
            f"  W  = [{W[0]:+.3f}, {W[1]:+.3f}, {W[2]:+.3f},\n"
            f"         {W[3]:+.3f}, {W[4]:+.3f}, {W[5]:+.3f}]\n"
            f"  x1 = {self.x[0,0]:+.5f}   x2 = {self.x[1,0]:+.5f}"
        )


# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    app = QApplication(sys.argv)
    win = LQR_RL_App()
    win.showMaximized()
    sys.exit(app.exec())